In [1]:
# ===== 1. ЗАГРУЗКА ДАННЫХ =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score, 
                            roc_auc_score, accuracy_score, confusion_matrix, roc_curve)

print("="*70)
print("ПОЛНЫЙ ПРОЦЕСС: ОТ ЗАГРУЗКИ ДАННЫХ ДО ЭКСПЕРИМЕНТА")
print("="*70)

# Загружаем данные
url_train = "https://raw.githubusercontent.com/dvigatelizm/keis7/main/train.csv"
print(f"Загружаем данные из: {url_train}")
df = pd.read_csv(url_train)

print(f"\nРазмер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print(f"\nПервые 5 строк:")
print(df.head())

# ===== 2. ПРЕДОБРАБОТКА ДАННЫХ =====
print("\n\n2. ПРЕДОБРАБОТКА ДАННЫХ:")
print("-"*40)

# Убираем шумные флаги, ID и Type
drop_cols = ['id', 'RNF', 'Type']  # оставляем TWF, HDF, PWF, OSF
print(f"Удаляем колонки: {drop_cols}")
df_clean = df.drop(columns=drop_cols)

print(f"Размер после очистки: {df_clean.shape}")
print(f"Оставшиеся колонки: {df_clean.columns.tolist()}")

# ===== 3. РАЗДЕЛЕНИЕ НА ПРИЗНАКИ И ЦЕЛЕВУЮ ПЕРЕМЕННУЮ =====
X = df_clean.drop(columns=['Machine failure'])
y = df_clean['Machine failure']

print(f"\nПризнаки (X): {X.shape}")
print(f"Целевая переменная (y): {y.shape}")
print(f"\nРаспределение целевой переменной:")
print(y.value_counts())
print(f"\nДоля положительного класса: {y.mean():.3f}")

# ===== 4. РАЗДЕЛЕНИЕ НА TRAIN/VAL ПО PRODUCT ID =====
print("\n\n4. РАЗДЕЛЕНИЕ НА TRAIN/VAL:")
print("-"*40)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=df['Product ID']))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

print(f"Train размер: {X_train.shape}")
print(f"Validation размер: {X_val.shape}")
print(f"\nРаспределение в train: {y_train.value_counts().to_dict()}")
print(f"Распределение в val: {y_val.value_counts().to_dict()}")

# ===== 5. МАСШТАБИРОВАНИЕ ЧИСЛОВЫХ ПРИЗНАКОВ =====
print("\n\n5. МАСШТАБИРОВАНИЕ ПРИЗНАКОВ:")
print("-"*40)

# Определяем числовые признаки
num_cols = ['Air temperature [K]', 'Process temperature [K]', 
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

print(f"Числовые признаки для масштабирования ({len(num_cols)}):")
for col in num_cols:
    print(f"  - {col}")

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

print("\nМасштабирование выполнено")
print("Пример данных после масштабирования:")
print(X_train[num_cols].head())

# ===== 6. СОЗДАЕМ КОПИИ БЕЗ PRODUCT ID =====
print("\n\n6. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛЕЙ:")
print("-"*40)

# Убираем Product ID для обучения моделей
X_train_dt = X_train.drop(columns=['Product ID'])
X_val_dt = X_val.drop(columns=['Product ID'])

print(f"X_train_dt (без Product ID): {X_train_dt.shape}")
print(f"X_val_dt (без Product ID): {X_val_dt.shape}")
print(f"\nПризнаки для обучения: {X_train_dt.columns.tolist()}")

# ===== 7. ОПРЕДЕЛЯЕМ 4 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКА =====
print("\n\n7. ОПРЕДЕЛЕНИЕ ВАЖНЫХ ПРИЗНАКОВ:")
print("-"*40)

# Обучаем базовую модель на всех признаках
print("Обучаем Decision Tree для определения важности признаков...")
dt_base = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_base.fit(X_train_dt, y_train)

# Получаем важность признаков
feature_importance = dt_base.feature_importances_
features = X_train_dt.columns.tolist()

# Создаем DataFrame для анализа
importance_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\nВАЖНОСТЬ ВСЕХ ПРИЗНАКОВ:")
print(importance_df.to_string(index=False))

# Выбираем 4 наиболее важных признака
top_4_features = importance_df.head(4)['feature'].tolist()
print(f"\n4 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКА: {top_4_features}")
print(f"Их важность: {importance_df.head(4)['importance'].values}")

# Сохраняем базовую модель
dt_model = dt_base

# ===== 8. ЭКСПЕРИМЕНТ: МОДЕЛЬ БЕЗ 4 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКОВ =====
print("\n\n" + "="*70)
print("ЭКСПЕРИМЕНТ: МОДЕЛЬ НА ЧИСЛОВЫХ ПРИЗНАКАХ БЕЗ 4 НАИБОЛЕЕ ВАЖНЫХ")
print("="*70)

# Определяем типы признаков
print("\n1. АНАЛИЗ ТИПОВ ПРИЗНАКОВ:")
print("-"*40)

# Числовые признаки (уже определены)
print(f"Числовые признаки ({len(num_cols)}):")
for col in num_cols:
    print(f"  - {col}")

# Бинарные флаги (остальные признаки)
binary_flags = [col for col in X_train_dt.columns if col not in num_cols]
print(f"\nБинарные флаги ({len(binary_flags)}):")
for col in binary_flags:
    print(f"  - {col}")

# Какие из топ-4 признаков числовые, а какие бинарные?
num_in_top4 = [feature for feature in top_4_features if feature in num_cols]
binary_in_top4 = [feature for feature in top_4_features if feature in binary_flags]

print(f"\nИз 4 наиболее важных признаков:")
print(f"  Числовые: {num_in_top4} ({len(num_in_top4)} шт.)")
print(f"  Бинарные: {binary_in_top4} ({len(binary_in_top4)} шт.)")

# ===== 9. СОЗДАЕМ НАБОРЫ ДАННЫХ ДЛЯ ЭКСПЕРИМЕНТА =====
print("\n\n2. СОЗДАНИЕ НАБОРОВ ДАННЫХ ДЛЯ ЭКСПЕРИМЕНТА:")
print("-"*40)

# Вариант 1: ТОЛЬКО числовые признаки (без тех, что в топ-4)
numeric_features_except_top4 = [f for f in num_cols if f not in top_4_features]
print(f"Вариант 1 - Только числовые (без топ-4):")
print(f"  Признаков: {len(numeric_features_except_top4)}")
if numeric_features_except_top4:
    print(f"  Список: {numeric_features_except_top4}")
else:
    print(f"  Все числовые признаки входят в топ-4!")

# Вариант 2: Все числовые признаки
print(f"\nВариант 2 - Все числовые признаки:")
print(f"  Признаков: {len(num_cols)}")
print(f"  Список: {num_cols}")

# Вариант 3: Все признаки КРОМЕ топ-4
all_except_top4 = [f for f in X_train_dt.columns if f not in top_4_features]
print(f"\nВариант 3 - Все признаки кроме топ-4:")
print(f"  Признаков: {len(all_except_top4)}")
print(f"  Список: {all_except_top4}")

# Вариант 4: Только бинарные флаги (без тех, что в топ-4)
binary_except_top4 = [f for f in binary_flags if f not in top_4_features]
print(f"\nВариант 4 - Только бинарные флаги (без топ-4):")
print(f"  Признаков: {len(binary_except_top4)}")
print(f"  Список: {binary_except_top4}")

# ===== 10. ФУНКЦИЯ ДЛЯ ОБУЧЕНИЯ И ОЦЕНКИ =====
def train_and_evaluate_model(features_list, model_name):
    """Обучает и оценивает модель на заданных признаках"""
    print(f"\n{model_name}:")
    print("-" * 40)
    
    # Проверяем признаки
    if len(features_list) == 0:
        print("  Нет признаков для обучения! Пропускаем.")
        return None, None
    
    # Создаем наборы данных
    X_train_subset = X_train_dt[features_list].copy()
    X_val_subset = X_val_dt[features_list].copy()
    
    print(f"  Признаки ({len(features_list)}): {features_list}")
    print(f"  Размер train: {X_train_subset.shape}")
    print(f"  Размер val: {X_val_subset.shape}")
    
    # Создаем и обучаем модель с теми же параметрами
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_train_subset, y_train)
    
    # Делаем предсказания
    y_val_pred = model.predict(X_val_subset)
    y_val_proba = model.predict_proba(X_val_subset)[:, 1]
    
    # Вычисляем метрики
    metrics = {
        'accuracy': accuracy_score(y_val, y_val_pred),
        'precision': precision_score(y_val, y_val_pred),
        'recall': recall_score(y_val, y_val_pred),
        'f1': f1_score(y_val, y_val_pred),
        'roc_auc': roc_auc_score(y_val, y_val_proba)
    }
    
    # Выводим метрики
    print("  Метрики:")
    for metric, value in metrics.items():
        print(f"    {metric.capitalize()}: {value:.4f}")
    
    print(f"  Глубина дерева: {model.get_depth()}")
    print(f"  Листьев: {model.get_n_leaves()}")
    
    return model, metrics

# ===== 11. ОБУЧАЕМ РАЗНЫЕ МОДЕЛИ =====
print("\n\n3. ОБУЧЕНИЕ И СРАВНЕНИЕ МОДЕЛЕЙ:")
print("="*70)

models = {}
metrics_results = {}

# Модель A: 4 наиболее важных признака (базовая)
print("\n" + "="*50)
print("МОДЕЛЬ A: 4 наиболее важных признака (БАЗОВАЯ)")
print("="*50)
models['A_top4'], metrics_results['A_top4'] = train_and_evaluate_model(
    top_4_features, "Модель A (Базовая)"
)

# Модель B: Все числовые признаки
print("\n" + "="*50)
print("МОДЕЛЬ B: Все числовые признаки")
print("="*50)
models['B_all_numeric'], metrics_results['B_all_numeric'] = train_and_evaluate_model(
    num_cols, "Модель B"
)

# Модель C: Числовые признаки без топ-4 (если есть)
print("\n" + "="*50)
print("МОДЕЛЬ C: Числовые признаки без топ-4")
print("="*50)
if numeric_features_except_top4:
    models['C_numeric_except_top4'], metrics_results['C_numeric_except_top4'] = train_and_evaluate_model(
        numeric_features_except_top4, "Модель C"
    )
else:
    print("  Все числовые признаки входят в топ-4, модель не создается")

# Модель D: Все признаки кроме топ-4
print("\n" + "="*50)
print("МОДЕЛЬ D: Все признаки кроме топ-4")
print("="*50)
models['D_all_except_top4'], metrics_results['D_all_except_top4'] = train_and_evaluate_model(
    all_except_top4, "Модель D"
)

# Модель E: Только бинарные флаги (без топ-4)
print("\n" + "="*50)
print("МОДЕЛЬ E: Только бинарные флаги без топ-4")
print("="*50)
if binary_except_top4:
    models['E_binary_except_top4'], metrics_results['E_binary_except_top4'] = train_and_evaluate_model(
        binary_except_top4, "Модель E"
    )
else:
    print("  Все бинарные признаки входят в топ-4, модель не создается")

# ===== 12. СРАВНИТЕЛЬНЫЙ АНАЛИЗ =====
print("\n\n4. СРАВНИТЕЛЬНЫЙ АНАЛИЗ:")
print("="*70)

# Создаем таблицу сравнения
comparison_data = []
for model_key in ['A_top4', 'B_all_numeric', 'C_numeric_except_top4', 
                  'D_all_except_top4', 'E_binary_except_top4']:
    if model_key in metrics_results:
        model_info = {
            'A_top4': ('4 важных признака', len(top_4_features)),
            'B_all_numeric': ('Все числовые', len(num_cols)),
            'C_numeric_except_top4': ('Числовые без топ-4', len(numeric_features_except_top4)),
            'D_all_except_top4': ('Все кроме топ-4', len(all_except_top4)),
            'E_binary_except_top4': ('Бинарные без топ-4', len(binary_except_top4))
        }[model_key]
        
        metrics = metrics_results[model_key]
        comparison_data.append({
            'Модель': model_info[0],
            'Признаков': model_info[1],
            'Accuracy': f"{metrics['accuracy']:.4f}",
            'Precision': f"{metrics['precision']:.4f}",
            'Recall': f"{metrics['recall']:.4f}",
            'F1': f"{metrics['f1']:.4f}",
            'ROC-AUC': f"{metrics['roc_auc']:.4f}"
        })

comparison_df = pd.DataFrame(comparison_data)
print("\nТАБЛИЦА СРАВНЕНИЯ МОДЕЛЕЙ:")
print(comparison_df.to_string(index=False))



ПОЛНЫЙ ПРОЦЕСС: ОТ ЗАГРУЗКИ ДАННЫХ ДО ЭКСПЕРИМЕНТА
Загружаем данные из: https://raw.githubusercontent.com/dvigatelizm/keis7/main/train.csv

Размер датасета: (136429, 14)
Колонки: ['id', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Первые 5 строк:
   id Product ID Type  Air temperature [K]  Process temperature [K]  \
0   0     L50096    L                300.6                    309.6   
1   1     M20343    M                302.6                    312.1   
2   2     L49454    L                299.3                    308.5   
3   3     L53355    L                301.0                    310.9   
4   4     M24050    M                298.0                    309.0   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1596         36.1              140                0    0   
1                    1

C:\Users\Сергей\AppData\Local\Temp\ipykernel_8172\3161462764.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
C:\Users\Сергей\AppData\Local\Temp\ipykernel_8172\3161462764.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[num_cols] = scaler.transform(X_val[num_cols])



ВАЖНОСТЬ ВСЕХ ПРИЗНАКОВ:
                feature  importance
                    HDF    0.426525
                    OSF    0.292741
                    PWF    0.159497
                    TWF    0.111044
            Torque [Nm]    0.007123
Process temperature [K]    0.001263
 Rotational speed [rpm]    0.001122
    Air temperature [K]    0.000361
        Tool wear [min]    0.000324

4 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКА: ['HDF', 'OSF', 'PWF', 'TWF']
Их важность: [0.42652495 0.29274052 0.15949701 0.11104407]


ЭКСПЕРИМЕНТ: МОДЕЛЬ НА ЧИСЛОВЫХ ПРИЗНАКАХ БЕЗ 4 НАИБОЛЕЕ ВАЖНЫХ

1. АНАЛИЗ ТИПОВ ПРИЗНАКОВ:
----------------------------------------
Числовые признаки (5):
  - Air temperature [K]
  - Process temperature [K]
  - Rotational speed [rpm]
  - Torque [Nm]
  - Tool wear [min]

Бинарные флаги (4):
  - TWF
  - HDF
  - PWF
  - OSF

Из 4 наиболее важных признаков:
  Числовые: [] (0 шт.)
  Бинарные: ['HDF', 'OSF', 'PWF', 'TWF'] (4 шт.)


2. СОЗДАНИЕ НАБОРОВ ДАННЫХ ДЛЯ ЭКСПЕРИМЕНТА:
-------------------